In [ ]:
!pip install yfinance pandas numpy scikit-learn xgboost shap matplotlib -q

In [ ]:
import os, numpy as np, pandas as pd, yfinance as yf, matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, roc_auc_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import shap
import warnings
warnings.filterwarnings("ignore")

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)


In [ ]:
# download data
path = "data/raw/spx_daily.csv"
if os.path.exists(path):
    spx = pd.read_csv(path, index_col="Date", parse_dates=True)
else:
    spx = yf.download("^GSPC", start="2005-01-01", end="2025-12-31", auto_adjust=True)
    spx.to_csv(path)

path = "data/raw/vix_term_structure.csv"
if os.path.exists(path):
    vix_raw = pd.read_csv(path, index_col="Date", parse_dates=True)
else:
    frames = {}
    for label, ticker in {"VIX": "^VIX", "VIX3M": "^VIX3M", "VIX6M": "^VIX6M"}.items():
        try:
            df = yf.download(ticker, start="2005-01-01", end="2025-12-31", auto_adjust=True)
            if len(df) > 0: frames[label] = df["Close"].squeeze()
        except: pass
    vix_raw = pd.DataFrame(frames)
    vix_raw.index.name = "Date"
    vix_raw.to_csv(path)

print(f"SPX: {len(spx)}, VIX: {len(vix_raw)}")


In [ ]:
# build all 24 features
close = spx["Close"].squeeze()
high = spx["High"].squeeze()
low = spx["Low"].squeeze()
volume = spx["Volume"].squeeze()
log_ret = np.log(close / close.shift(1))
squared = log_ret ** 2

feat = pd.DataFrame(index=close.index)

# P5 features
for h in [1, 5, 22]:
    feat[f"RV_{h}d"] = np.sqrt(squared.rolling(h).sum() * (252 / h))

# returns
feat["log_return"] = log_ret
feat["abs_return"] = log_ret.abs()
feat["return_5d"] = close.pct_change(5)
feat["return_22d"] = close.pct_change(22)

# vol measures
feat["vol_5d"] = log_ret.rolling(5).std() * np.sqrt(252)
feat["vol_22d"] = log_ret.rolling(22).std() * np.sqrt(252)
feat["parkinson_vol"] = np.sqrt((1 / (4 * np.log(2))) * (np.log(high / low) ** 2)) * np.sqrt(252)

# volume and momentum
feat["volume_zscore"] = (volume - volume.rolling(22).mean()) / volume.rolling(22).std()
delta = close.diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss_s = (-delta.clip(upper=0)).rolling(14).mean()
feat["rsi_14"] = 100 - (100 / (1 + gain / loss_s))

# VIX features
feat["vix_level"] = vix_raw.get("VIX")
if "VIX" in vix_raw.columns and "VIX3M" in vix_raw.columns:
    feat["slope_3m_spot"] = vix_raw["VIX3M"] - vix_raw["VIX"]
if "VIX3M" in vix_raw.columns and "VIX6M" in vix_raw.columns:
    feat["slope_6m_3m"] = vix_raw["VIX6M"] - vix_raw["VIX3M"]
if all(c in vix_raw.columns for c in ["VIX", "VIX3M", "VIX6M"]):
    feat["curvature"] = vix_raw["VIX"] - 2 * vix_raw["VIX3M"] + vix_raw["VIX6M"]
if "VIX" in vix_raw.columns and "VIX3M" in vix_raw.columns:
    feat["contango_flag"] = (vix_raw["VIX3M"] > vix_raw["VIX"]).astype(int)
if "VIX" in vix_raw.columns:
    feat["vix_1d_change"] = vix_raw["VIX"].pct_change()
    feat["vix_zscore_20d"] = (
        (vix_raw["VIX"] - vix_raw["VIX"].rolling(20).mean()) / vix_raw["VIX"].rolling(20).std()
    )
    feat["vix_ma_ratio"] = vix_raw["VIX"] / vix_raw["VIX"].rolling(60).mean()

# flow proxies
if "vix_level" in feat.columns:
    feat["vix_momentum_5d"] = feat["vix_level"].pct_change(5)
    feat["vix_momentum_22d"] = feat["vix_level"].pct_change(22)
    feat["vix_acceleration"] = feat["vix_momentum_5d"].diff(5)
if "slope_3m_spot" in feat.columns:
    feat["term_slope_change_5d"] = feat["slope_3m_spot"].diff(5)

# targets
fwd_rv = []
sq = squared.values
for i in range(len(sq)):
    end = i + 1 + 22
    if end > len(sq):
        fwd_rv.append(np.nan)
    else:
        fwd_rv.append(np.sqrt(np.sum(sq[i+1:end]) * (252 / 22)))

feat["fwd_rv_22d"] = fwd_rv
feat = feat.dropna()
threshold = np.percentile(feat["fwd_rv_22d"], 90)
feat["spike_label"] = (feat["fwd_rv_22d"] > threshold).astype(int)

target_cols = ["fwd_rv_22d", "spike_label"]
feature_cols = [c for c in feat.columns if c not in target_cols]

print(f"Dataset: {len(feat)} rows, {len(feature_cols)} features")


In [ ]:
# feature overview plots
fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=True)

axes[0].plot(feat.index, feat["log_return"], linewidth=0.4, alpha=0.7)
axes[0].set_title("Daily Log Returns")

axes[1].plot(feat.index, feat["vol_22d"], linewidth=0.7, label="Close-to-close")
axes[1].plot(feat.index, feat["parkinson_vol"], linewidth=0.5, alpha=0.7, label="Parkinson")
axes[1].set_title("Volatility Comparison"); axes[1].legend()

axes[2].plot(feat.index, feat["volume_zscore"], linewidth=0.5, color="green")
axes[2].set_title("Volume Z-Score")

axes[3].plot(feat.index, feat["rsi_14"], linewidth=0.5, color="purple")
axes[3].axhline(70, color="red", linestyle="--", alpha=0.5)
axes[3].axhline(30, color="green", linestyle="--", alpha=0.5)
axes[3].set_title("RSI (14-day)")

plt.tight_layout()
plt.show()


In [ ]:
# what a single transformer input window looks like (60 days ending before COVID)
target_date = "2020-02-19"
idx = feat.index.get_indexer([pd.Timestamp(target_date)], method="nearest")[0]
window = feat.iloc[idx-59:idx+1][feature_cols]
window_norm = (window - window.min()) / (window.max() - window.min() + 1e-10)

fig, ax = plt.subplots(figsize=(16, 10))
ax.imshow(window_norm.T, aspect="auto", cmap="YlOrRd", interpolation="nearest")
ax.set_yticks(range(len(feature_cols)))
ax.set_yticklabels(feature_cols, fontsize=7)
ax.set_xlabel("Day in window (0=oldest, 59=most recent)")
ax.set_title(f"60-Day Window Ending {window.index[-1].date()}")
plt.tight_layout()
plt.show()


In [ ]:
# train/test split
har_cols = ["RV_1d", "RV_5d", "RV_22d"]
train = feat[feat.index < "2020-01-01"]
test = feat[feat.index >= "2020-01-01"]

X_train, X_test = train[feature_cols], test[feature_cols]
y_train = train["fwd_rv_22d"]
y_test = test["fwd_rv_22d"]
y_test_cls = test["spike_label"]

# HAR-RV baseline
har = LinearRegression().fit(X_train[har_cols], y_train)
har_pred = har.predict(X_test[har_cols])

# XGBoost on all features
xgb_model = xgb.XGBRegressor(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.7,
    reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbosity=0,
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

print("HAR-RV:")
print(f"  MSE={mean_squared_error(y_test, har_pred):.6f}  AUC={roc_auc_score(y_test_cls, har_pred):.4f}")
print("XGBoost (all features):")
print(f"  MSE={mean_squared_error(y_test, xgb_pred):.6f}  AUC={roc_auc_score(y_test_cls, xgb_pred):.4f}")


In [ ]:
# SHAP by regime
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

median_rv = y_test.median()
calm = y_test <= median_rv
stress = y_test > median_rv

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

plt.sca(axes[0])
shap.summary_plot(shap_values[calm.values], X_test[calm], plot_type="bar", show=False, max_display=15)
axes[0].set_title(f"Calm (fwd_rv <= {median_rv:.3f})")

plt.sca(axes[1])
shap.summary_plot(shap_values[stress.values], X_test[stress], plot_type="bar", show=False, max_display=15)
axes[1].set_title(f"Stress (fwd_rv > {median_rv:.3f})")

plt.suptitle("SHAP by Regime")
plt.tight_layout()
plt.show()


In [ ]:
# meta-model: add autoencoder features if available
latent_path = "data/processed/p1_latent_features.csv"
if os.path.exists(latent_path):
    latent_df = pd.read_csv(latent_path, index_col=0, parse_dates=True)
    meta = feat[feature_cols].join(latent_df, how="inner")
    meta = meta.join(feat[target_cols], how="inner").dropna()

    meta_feat_cols = [c for c in meta.columns if c not in target_cols]
    m_train = meta[meta.index < "2020-01-01"]
    m_test = meta[meta.index >= "2020-01-01"]

    xgb_meta = xgb.XGBRegressor(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7,
        reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbosity=0,
    )
    xgb_meta.fit(m_train[meta_feat_cols], m_train["fwd_rv_22d"])
    meta_pred = xgb_meta.predict(m_test[meta_feat_cols])

    print(f"Meta-model ({len(meta_feat_cols)} features):")
    print(f"  MSE={mean_squared_error(m_test['fwd_rv_22d'], meta_pred):.6f}")
    print(f"  AUC={roc_auc_score(m_test['spike_label'], meta_pred):.4f}")

    exp = shap.TreeExplainer(xgb_meta)
    sv = exp.shap_values(m_test[meta_feat_cols])
    plt.figure(figsize=(10, 10))
    shap.summary_plot(sv, m_test[meta_feat_cols], plot_type="bar", show=False, max_display=25)
    plt.title("SHAP — Meta-Model with Autoencoder Features")
    plt.tight_layout()
    plt.show()
else:
    print("Run the P1 notebook first to generate latent features.")


In [ ]:
# save P2 features
feat[feature_cols].to_csv("data/processed/p2_features.csv")
print("Saved p2_features.csv")
